# 05. 팀 취합 단계 검증 (Team Verification)

README에 들어간 수치 중 취합 단계에서 새로 측정한 것(그룹 교차검증, 변수 묶음 제거, 불균형 처리 반복 비교, 6종 분류, Y 좌표 제거, 데이터 구조)을 재현하는 노트북입니다. README 8.3절 순열 중요도는 `07_final_model.ipynb`의 6-2 셀에 있습니다.

| 절 | 내용 | README | 사용 데이터 |
|---|---|---|---|
| 1 | 데이터 사실 확인 | 5.2, 6.2절 | Train (유사 중복은 전체) |
| 2 | 생산 조건 묶음 구조 | 6.1절 발견 2 | Train · Test (조건 겹침 확인) |
| 3 | 모양만으로 구분되는 정도 | 6.1절 발견 3 | 전체 |
| 4 | 롤 마크 주기 | 6.1절 발견 4 | 전체 |
| 5 | 모델 정의 | — | — |
| 6 | 일반 CV vs 그룹 CV | 1.3, 7.4, 7.5, 8.4절 | Train |
| 7 | 불균형 처리 반복 비교 | 7.3절 | Train |
| 8 | 정답 체계와 Y 좌표 | 8.2, 8.4절 | Train |

- 모델 선택에 쓰는 실험(6~8절)은 모두 **Train 데이터(1,552건)** 안에서만 수행합니다.
- 데이터 구조 분석(2~4절)은 README에 적힌 대로 Test도 함께 쓰며, 이 결과는 모델이나 설정을 고르는 데 쓰지 않았습니다.
- 최종 모델의 파생변수·하이퍼파라미터·학습 코드는 `07_final_model.ipynb`와 같습니다. 실행 환경 차이로 소수점 셋째 자리에서 값이 달라질 수 있으므로, 비교는 항상 **이 노트북 안에서 같은 조건끼리** 합니다.
- 반복 교차검증은 `07_final_model.ipynb`와 같은 `RepeatedStratifiedKFold(5, 3, random_state=42)`, 그룹 교차검증은 `StratifiedGroupKFold(5)`를 시드 세 개(42, 0, 1)로 돌려 똑같이 15 fold로 맞췄습니다.
- 전체 실행 시간: 1코어 기준 약 1시간 30분 (오래 걸리는 셀은 제목에 표시했습니다).

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, RepeatedStratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import kruskal
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler

RS = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# 팀 공통 전처리 데이터와 Train/Test 인덱스 (07_final_model.ipynb와 같은 파일)
df = pd.read_csv(ROOT / "data/processed/Faults_processed.csv")
tr = pd.read_csv(ROOT / "data/split/train_idx.csv")["index"].to_numpy()
te = pd.read_csv(ROOT / "data/split/test_idx.csv")["index"].to_numpy()
print("Train", len(tr), "| Test", len(te))
print(df.loc[tr, "target"].value_counts().to_dict())

## 1. 데이터 사실 확인 (README 5.2, 6.2절)

In [ ]:
raw = pd.read_csv(ROOT / "data/raw/Faults.csv")
T = df.loc[tr]
print("결함 오른쪽 끝 <= Length_of_Conveyer 인 행 비율:", (raw.X_Maximum <= raw.Length_of_Conveyer).mean())
print("Pixels_Areas - Sum_of_Luminosity Spearman (Train):",
      round(T[["Pixels_Areas", "Sum_of_Luminosity"]].corr(method="spearman").iloc[0, 1], 3))
print("TypeOfSteel_A300 + TypeOfSteel_A400 값:", (raw.TypeOfSteel_A300 + raw.TypeOfSteel_A400).unique())

# 유사 중복 의심 쌍과 분할 위치
train_set = set(tr)
for i in [1775, 1776, 529, 678]:
    print(i, "Train" if i in train_set else "Test")
pair = df.loc[[1775, 1776]].T
display(pair[pair[1775] != pair[1776]])

In [ ]:
# 극단값 구성 (Train, 면적 사분위 범위 1.5배 밖)
q1, q3 = T.Pixels_Areas.quantile([.25, .75])
m = T.Pixels_Areas > q3 + 1.5 * (q3 - q1)
print("극단값 수:", m.sum(), " 그중 K_Scratch 비율:", round((T.target[m] == "K_Scratch").mean(), 3))

# Kruskal-Wallis (Train)
kw = []
for c in T.columns.drop("target"):
    H, p = kruskal(*[T.loc[T.target == k, c] for k in T.target.unique()])
    kw.append((c, (H - 6) / (len(T) - 7), p))
display(pd.DataFrame(kw, columns=["변수", "효과크기 ε²", "p"]).sort_values("효과크기 ε²", ascending=False))

## 2. 생산 조건 묶음 구조 (README 6.1절 발견 2)

In [ ]:
df["grp"] = (df.TypeOfSteel_A300.astype(str) + "_" + df.Steel_Plate_Thickness.astype(str)
             + "_" + df.Length_of_Conveyer.astype(str))
T, Te = df.loc[tr], df.loc[te]

def purity(d):
    n = d.groupby("grp").size()
    c = d.groupby("grp").target.agg(lambda s: s.value_counts().iloc[0] / len(s))
    return (c * n).sum() / n.sum()

rng = np.random.default_rng(0); sh = []
for _ in range(200):
    t = T.copy(); t["target"] = rng.permutation(t.target.values); sh.append(purity(t))
n = T.groupby("grp").size()
print("묶음 수:", len(n), " 중앙값:", n.median(), " 최대:", n.max())
print("실제 순도:", round(purity(T), 3), " 무작위 평균/최대:", round(np.mean(sh), 3), round(np.max(sh), 3))
print("Test 행 중 Train에 같은 묶음이 있는 비율:", round(Te.grp.isin(set(T.grp)).mean(), 3))
print("Test 행 중 처음 보는 묶음:", (~Te.grp.isin(set(T.grp))).sum(), "건")

## 3. 모양만으로 구분되는 정도 (README 6.1절 발견 3)

In [ ]:
SHAPE = ["LogOfAreas", "Log_X_Index", "Log_Y_Index", "Orientation_Index", "Square_Index", "Empty_Index",
         "Edges_X_Index", "Edges_Y_Index", "Luminosity_Index", "Minimum_of_Luminosity", "Maximum_of_Luminosity"]
PROC = ["TypeOfSteel_A300", "Steel_Plate_Thickness", "Length_of_Conveyer"]
lab = df.target.to_numpy()

def neighbor_ratio(cols):
    Z = StandardScaler().fit_transform(df[cols])
    _, ind = NearestNeighbors(n_neighbors=6).fit(Z).kneighbors(Z)      # 자기 자신 + 이웃 5개
    return pd.Series((lab[ind[:, 1:]] == lab[:, None]).mean(1)).groupby(lab).mean()

display(pd.DataFrame({"모양·밝기만": neighbor_ratio(SHAPE), "+ 공정 조건": neighbor_ratio(SHAPE + PROC)}).round(3))

## 4. 롤 마크 주기 (README 6.1절 발견 4) — 약 3분

In [ ]:
df["xc"] = (df.X_Minimum + df.X_Maximum) / 2
col = df[(df.Length_of_Conveyer == 1358) & (df.Steel_Plate_Thickness == 40)
         & (df.TypeOfSteel_A300 == 0) & (abs(df.xc - 125) <= 20)]
yv = col.Y_Minimum.values

def fit_rate(yy, p, tol=15):
    d = np.diff(np.sort(yy)); d = d[d > 0]; r = d / p
    return np.mean(np.abs(r - np.round(r)) * p <= tol)

best_s, best_p = max((fit_rate(yv, p), p) for p in np.arange(1000, 20000, 0.5))
print(len(col), col.target.value_counts().to_dict(), " 최적 주기:", best_p, " 적합률:", round(best_s, 3))
rng = np.random.default_rng(0)
fixed = np.mean([fit_rate(rng.uniform(yv.min(), yv.max(), len(yv)), best_p) for _ in range(200)])
search = np.mean([max(fit_rate(yy, p) for p in np.arange(1000, 20000, 2))
                  for yy in [rng.uniform(yv.min(), yv.max(), len(yv)) for _ in range(20)]])
print("무작위(같은 주기):", round(fixed, 3), " 무작위(주기 다시 탐색):", round(search, 3))
print("Train/Test 분포:", pd.Series(["Train" if i in train_set else "Test" for i in col.index]).value_counts().to_dict())

## 5. 모델 정의

`add_features`와 세 구성 모델의 설정은 `07_final_model.ipynb`와 같습니다. `Final`의 `mode`로 입력 변수만 바꿔 가며 효과를 분리합니다.

In [ ]:
D = df.loc[tr].reset_index(drop=True)
X_train, y_train = D.drop(columns=["target", "grp", "xc"]), D["target"]
GRP = D["grp"].values

# ── 07_final_model.ipynb와 같은 파생변수 ──
PLATE_KEYS = ["Length_of_Conveyer", "Steel_Plate_Thickness", "TypeOfSteel_A300"]
DISCRETE = ["TypeOfSteel_A300", "Outside_Global_Index"]

def plate_key(X):
    return X[PLATE_KEYS].astype(str).agg("_".join, axis=1)

def add_features(X, plate_counts):
    out = X.copy()
    width = (X["X_Maximum"] - X["X_Minimum"]).clip(lower=0)
    mx, mn = X["Maximum_of_Luminosity"], X["Minimum_of_Luminosity"]
    out["compactness"] = (X["X_Perimeter"] + X["Y_Perimeter"]) / np.sqrt(X["Pixels_Areas"])
    out["perimeter_ratio_log"] = np.log((X["X_Perimeter"] + 1) / (X["Y_Perimeter"] + 1))
    out["x_straightness"] = X["X_Perimeter"] / (width + 1)
    out["lum_range"] = mx - mn
    out["lum_contrast"] = (mx - mn) / (mx + mn).replace(0, 1)
    out["lum_mean_position"] = (X["Sum_of_Luminosity"] / X["Pixels_Areas"] - mn) / (mx - mn + 1)
    xc = (X["X_Minimum"] + X["X_Maximum"]) / 2 / X["Length_of_Conveyer"]
    out["x_center_rel"] = xc
    out["x_center_dist"] = (xc - 0.5).abs()
    out["plate_defect_count"] = plate_key(X).map(plate_counts).fillna(0).astype(float)   # 처음 보는 조합은 0
    out["thickness_per_conveyer"] = X["Steel_Plate_Thickness"] / X["Length_of_Conveyer"]
    return out[[c for c in out.columns if c not in DISCRETE] + DISCRETE]

DROPPROC = PLATE_KEYS + ["plate_defect_count", "thickness_per_conveyer", "x_center_rel", "x_center_dist"]

class Final:
    """최종 모델 (ExtraTrees + RandomForest + GBM 소프트보팅)
    mode: full(36개) / nofe(원본 26개) / nopdc(plate_defect_count 제외) / noproc(공정 조건 제외) / proconly(공정 조건 3개)"""
    def __init__(self, mode="full"):
        self.mode = mode
    def prep(self, X):
        if self.mode == "nofe":
            return X.copy()
        if self.mode == "proconly":
            return X[PLATE_KEYS]
        Z = add_features(X, self.pc)
        if self.mode == "nopdc":
            Z = Z.drop(columns=["plate_defect_count"])
        if self.mode == "noproc":
            Z = Z.drop(columns=[c for c in DROPPROC if c in Z])
        return Z
    def fit(self, X, y):
        self.pc = plate_key(X).value_counts().to_dict()          # 학습 fold로만 집계
        Xf = self.prep(X)
        et = ExtraTreesClassifier(n_estimators=438, max_depth=15, max_features=0.7, min_samples_leaf=1,
                                  class_weight="balanced", random_state=RS, n_jobs=1)
        rf = RandomForestClassifier(n_estimators=483, max_depth=None, max_features=0.3, min_samples_leaf=1,
                                    random_state=RS, n_jobs=1)
        gbm = GradientBoostingClassifier(n_estimators=240, learning_rate=0.08793919399591556, max_depth=4,
                                         subsample=0.9079017872741743, max_features=0.5, min_samples_leaf=5,
                                         random_state=RS)
        counts = y.value_counts()
        et.fit(Xf, y); rf.fit(Xf, y)
        gbm.fit(Xf, y, sample_weight=y.map(counts.max() / counts).to_numpy())
        self.m = [et, rf, gbm]; self.classes_ = et.classes_
        return self
    def predict_proba(self, X):
        Xf = self.prep(X).replace([np.inf, -np.inf], np.nan).fillna(0)
        return np.mean([m.predict_proba(Xf) for m in self.m], axis=0)
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

class XGBBase:
    """Baseline: XGBoost 기본값"""
    def fit(self, X, y):
        self.cl = np.unique(y)
        self.m = XGBClassifier(random_state=RS, n_jobs=1, eval_metric="mlogloss").fit(X, np.searchsorted(self.cl, y))
        return self
    def predict(self, X):
        return self.cl[self.m.predict(X)]

class GBXGB_BSMOTE:
    """후보: GB + XGBoost 소프트보팅, 각 모델의 학습 fold 안에서만 BorderlineSMOTE"""
    def fit(self, X, y):
        self.cl = np.unique(y); yi = np.searchsorted(self.cl, y); self.ms = []
        for mk in [lambda: GradientBoostingClassifier(random_state=RS),
                   lambda: XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6,
                                         eval_metric="mlogloss", random_state=RS, n_jobs=1)]:
            Xa, ya = BorderlineSMOTE(k_neighbors=3, m_neighbors=5, random_state=RS).fit_resample(X.values, yi)
            self.ms.append(mk().fit(Xa, ya))
        return self
    def predict(self, X):
        return self.cl[np.mean([m.predict_proba(X.values) for m in self.ms], axis=0).argmax(1)]

def run_cv(make, kind="strat", X=None, y=None, groups=None, seeds=(42, 0, 1)):
    """strat: RepeatedStratifiedKFold(5×3, random_state=42) / group: StratifiedGroupKFold(5) × 시드 3개
    둘 다 15 fold. fold별 Macro F1 배열을 반환한다."""
    X = X_train if X is None else X
    y = y_train if y is None else y
    groups = GRP if groups is None else groups
    if kind == "strat":
        splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=len(seeds), random_state=RS).split(X, y))
    else:
        splits = [s for seed in seeds
                  for s in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(X, y, groups)]
    return np.array([f1_score(y.iloc[b], make().fit(X.iloc[a], y.iloc[a]).predict(X.iloc[b]), average="macro")
                     for a, b in splits])

## 6. 일반 교차검증 vs 그룹 교차검증 (README 1.3, 7.4, 7.5, 8.4절) — 약 60분

In [ ]:
variants = {
    "Baseline XGBoost": XGBBase,
    "최종 모델": lambda: Final(),
    "최종 모델 − 파생변수 제외": lambda: Final("nofe"),
    "최종 모델 − plate_defect_count": lambda: Final("nopdc"),
    "최종 모델 − 공정 조건 제외": lambda: Final("noproc"),
    "공정 조건 3개만": lambda: Final("proconly"),
    "GB+XGB+BorderlineSMOTE": GBXGB_BSMOTE,
}
res = {}
for name, mk in variants.items():
    t = time.time()
    res[name] = {k: run_cv(mk, k) for k in ["strat", "group"]}
    s, g = res[name]["strat"], res[name]["group"]
    print(f"{name:30s} 5-fold {s[:5].mean():.4f} | 반복 {s.mean():.4f}±{s.std():.4f} | "
          f"그룹 {g.mean():.4f}±{g.std():.4f}  ({time.time() - t:.0f}초)")

# 같은 fold끼리 짝 비교
for a, b in [("최종 모델", "Baseline XGBoost"), ("최종 모델", "최종 모델 − 파생변수 제외"),
             ("최종 모델 − plate_defect_count", "최종 모델")]:
    for k in ["strat", "group"]:
        d = res[a][k] - res[b][k]
        print(f"{a} − {b} [{k}]: 평균 {d.mean():+.4f}, 15번 중 {(d > 0).sum()}번 우세")

## 7. 불균형 처리 반복 비교 (README 7.3절) — 약 15분

GBM 기본값에 불균형 처리 방법만 바꿔 같은 15 fold에서 비교합니다.

In [ ]:
classes = np.unique(y_train)
yi = np.searchsorted(classes, y_train.values)
Xv = X_train.values

def fit_gb(method, Xa, ya):
    sw = None
    if method == "클래스 가중치":
        sw = compute_sample_weight("balanced", ya)
    elif method == "SMOTE":
        Xa, ya = SMOTE(k_neighbors=3, random_state=RS).fit_resample(Xa, ya)
    elif method == "BorderlineSMOTE":
        Xa, ya = BorderlineSMOTE(k_neighbors=3, m_neighbors=5, random_state=RS).fit_resample(Xa, ya)
    elif method == "언더샘플링":
        Xa, ya = RandomUnderSampler(random_state=RS).fit_resample(Xa, ya)
    return GradientBoostingClassifier(random_state=RS).fit(Xa, ya, sample_weight=sw)

imb, recall = {}, {}
for meth in ["처리 없음", "클래스 가중치", "BorderlineSMOTE", "SMOTE", "언더샘플링"]:
    sc = []
    for s in [42, 0, 1]:
        oof = np.zeros(len(yi), dtype=int)
        for a, b in StratifiedKFold(5, shuffle=True, random_state=s).split(Xv, yi):
            p = fit_gb(meth, Xv[a], yi[a]).predict(Xv[b]); oof[b] = p
            sc.append(f1_score(yi[b], p, average="macro"))
        if s == 42:   # 유형별 Recall은 시드 42 기준
            recall[meth] = [((oof == k) & (yi == k)).sum() / (yi == k).sum() for k in range(len(classes))]
    imb[meth] = np.array(sc); d = imb[meth] - imb["처리 없음"]
    print(f"{meth:15s} 시드42 {imb[meth][:5].mean():.4f} | 반복 {imb[meth].mean():.4f}±{imb[meth].std():.4f} | "
          f"차이 {d.mean():+.4f} | 우세 {(d > 0).sum()}/15")
display(pd.DataFrame(recall, index=classes).round(3))

## 8. 정답 체계와 Y 좌표 (README 8.2, 8.4절) — 약 15분

In [ ]:
# Other_Faults 제외 6종 분류 (5-fold)
keep = (y_train != "Other_Faults").values
six = run_cv(lambda: Final(), "strat", X=X_train[keep].reset_index(drop=True),
             y=y_train[keep].reset_index(drop=True), groups=GRP[keep])
print("Other_Faults 제외 6종: 5-fold", six[:5].mean().round(4), "| 반복", six.mean().round(4))

# Y 좌표 제거 (반복 평균)
XnoY = X_train.drop(columns=["Y_Minimum", "Y_Maximum"])
for k in ["strat", "group"]:
    print(f"Y 제외 [{k}]:", run_cv(lambda: Final(), k, X=XnoY).mean().round(4),
          " / Y 포함:", res["최종 모델"][k].mean().round(4))